

Implement a simple custom RNN cell that processes a sequence of at least 5 time steps ($T \ge 5$). Compute a scalar loss, call `.backward()`, and print the gradient tensor (`.grad`) for each model parameter. Then explain what those parameter gradients physically represent.

## 1. Imports and reproducibility

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)
print("PyTorch:", torch.__version__)

PyTorch: 2.8.0


## 2. Recurrence that I'm going to implement

A vanilla RNN updates a hidden state at every time step $t$ using the **same** trainable parameters:

$$
h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)
$$

- $x_t$: input at time $t$
- $h_{t-1}$: previous hidden state (memory)
- $W_{xh}$, $W_{hh}$, $b_h$: shared across all time steps

The initial state is $h_0 = 0$. After $T = 5$ steps we have $h_5$, and the scalar loss is taken only from that last hidden state:

$$
L = L(h_5)
$$

## 3. Custom RNN cell

`nn.Parameter` registers each tensor as a trainable weight so PyTorch will store `.grad` on it after `.backward()`.

In [6]:
class SimpleRNNCell(nn.Module):
    """One vanilla RNN step: h_t = tanh(W_xh x_t + W_hh h_{t-1} + b_h)."""

    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.W_xh = nn.Parameter(torch.randn(hidden_size, input_size) * 0.1)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.1)
        self.b_h = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:
        return torch.tanh(x_t @ self.W_xh.T + h_prev @ self.W_hh.T + self.b_h)


input_size = 3
hidden_size = 4
cell = SimpleRNNCell(input_size, hidden_size)

print("Trainable parameters:")
for name, param in cell.named_parameters():
    print(f"  {name:6}  shape={tuple(param.shape)}  requires_grad={param.requires_grad}")

Trainable parameters:
  W_xh    shape=(4, 3)  requires_grad=True
  W_hh    shape=(4, 4)  requires_grad=True
  b_h     shape=(4,)  requires_grad=True


## 4. Input sequence with $T = 5$

Each time step is a 3-dimensional feature vector. `h_0` starts as zeros.

In [7]:
T = 5
sequence = torch.randn(T, input_size)
h_0 = torch.zeros(hidden_size)

print(f"T = {T}  (requirement: T >= 5)")
print("sequence shape:", tuple(sequence.shape), "  ->  (time, input_size)")
print("h_0:", h_0)
print("\nInputs x_1 ... x_5:")
for t, x_t in enumerate(sequence, start=1):
    print(f"  x_{t} = {x_t.tolist()}")

T = 5  (requirement: T >= 5)
sequence shape: (5, 3)   ->  (time, input_size)
h_0: tensor([0., 0., 0., 0.])

Inputs x_1 ... x_5:
  x_1 = [0.3040127456188202, 0.1051730141043663, 0.960340142250061]
  x_2 = [-0.5671805739402771, -0.5706474184989929, 1.5980384349822998]
  x_3 = [0.11148621141910553, -0.03919669985771179, 1.4111539125442505]
  x_4 = [-0.655610978603363, 0.8576056957244873, -1.6270242929458618]
  x_5 = [-1.3951387405395508, -0.23872417211532593, -0.5049903988838196]


## 5. Forward pass (unroll the cell through time)

The same cell — and therefore the same $W_{xh}$, $W_{hh}$, $b_h$ — is applied at every step:

$$
h_1 = f(x_1, h_0),\quad
h_2 = f(x_2, h_1),\quad
\ldots,\quad
h_5 = f(x_5, h_4)
$$

In [8]:
h_t = h_0
hidden_states = []

for t, x_t in enumerate(sequence, start=1):
    h_t = cell(x_t, h_t)
    hidden_states.append(h_t)
    print(f"h_{t} = {h_t.detach().tolist()}")

h_T = hidden_states[-1]
print(f"\nTerminal hidden state h_{T} shape: {tuple(h_T.shape)}")

h_1 = [-0.04547335207462311, 0.043566230684518814, 0.020723409950733185, 0.023178184404969215]
h_2 = [0.023704569786787033, 0.0409311018884182, 0.11656846106052399, -0.14595039188861847]
h_3 = [-0.0351826511323452, 0.08056643605232239, 0.0563761442899704, -0.015674816444516182]
h_4 = [0.12163402140140533, 0.043391596525907516, -0.0999240055680275, -0.07964208722114563]
h_5 = [0.09421073645353317, 0.0099510932341218, 0.023548197001218796, -0.22985738515853882]

Terminal hidden state h_5 shape: (4,)


## 6. Scalar loss from the last hidden state

Loss is a single number $L = L(h_T)$. Here we use mean squared error against a dummy target. A scalar is required so `.backward()` can start from one value and send gradients back through all five time steps.

In [10]:
target = torch.zeros_like(h_T)
loss = ((h_T - target) ** 2).mean()

print("target:", target.tolist())
print("h_T:   ", h_T.detach().tolist())
print(f"scalar loss L = {loss.item():.6f}")
print("loss is a 0-dim tensor:", tuple(loss.shape), "  requires_grad:", loss.requires_grad)

target: [0.0, 0.0, 0.0, 0.0]
h_T:    [0.09421073645353317, 0.0099510932341218, 0.023548197001218796, -0.22985738515853882]
scalar loss L = 0.015591
loss is a 0-dim tensor: ()   requires_grad: True


## 7. Backpropagation Through Time

`loss.backward()` walks the unrolled graph

$$
L \leftarrow h_5 \leftarrow h_4 \leftarrow h_3 \leftarrow h_2 \leftarrow h_1
$$

and **adds** the contribution of every time step into the same shared parameters. That accumulated tensor is stored in `.grad`.

In [11]:
print("Gradients before backward (should be None):")
for name, param in cell.named_parameters():
    print(f"  {name}.grad = {param.grad}")

loss.backward()
print("\nCalled loss.backward()")

Gradients before backward (should be None):
  W_xh.grad = None
  W_hh.grad = None
  b_h.grad = None

Called loss.backward()


## 8. Gradient tensor for each parameter

In [13]:
for name, param in cell.named_parameters():
    print(f"\n{name}.grad  shape={tuple(param.grad.shape)}")
    print(param.grad)


W_xh.grad  shape=(4, 3)
tensor([[-0.0550, -0.0231,  0.0028],
        [-0.0002, -0.0101,  0.0136],
        [-0.0198,  0.0012, -0.0163],
        [ 0.1497,  0.0289,  0.0500]])

W_hh.grad  shape=(4, 4)
tensor([[ 0.0063,  0.0010, -0.0051, -0.0039],
        [ 0.0010, -0.0006, -0.0011, -0.0002],
        [ 0.0012,  0.0008, -0.0011, -0.0007],
        [-0.0134, -0.0044,  0.0111,  0.0086]])

b_h.grad  shape=(4,)
tensor([ 0.0348, -0.0059,  0.0144, -0.1051])


## 9. What these parameter gradients physically represent

For any weight $\theta$, the stored tensor is the partial derivative of the scalar loss:

$$
\theta.\mathrm{grad} = \frac{\partial L}{\partial \theta}
$$

Each entry answers: **if I increase this one number in the model by a tiny amount, how much does $L$ change?**

| Parameter | Shape | Meaning of its `.grad` |
|---|---|---|
| $W_{xh}$ | `(hidden, input)` | Sensitivity of $L$ to the **input-to-hidden** map. Entry $(i, j)$ says how changing the weight from input feature $j$ into hidden unit $i$ moves the loss. |
| $W_{hh}$ | `(hidden, hidden)` | Sensitivity of $L$ to the **recurrent / memory** map. Entry $(i, k)$ says how changing the connection from previous hidden unit $k$ into hidden unit $i$ moves the loss. |
| $b_h$ | `(hidden,)` | Sensitivity of $L$ to each hidden unit's **bias** (a constant shift before $\tanh$). |

Sign and size:

- **Positive** gradient: increasing that weight **raises** $L$ (worse, for this MSE).
- **Negative** gradient: increasing that weight **lowers** $L$.
- **Larger magnitude**: the loss is more sensitive to that weight.

Because the cell is reused at $t = 1,\ldots,5$, each `.grad` is **not** from a single step. It is the **sum of five contributions** along the unrolled chain (BPTT). That is why a change in $W_{hh}$ matters twice: it changes $h_t$ now, and it also changes every later hidden state that depends on $h_t$.

An optimizer would then take a step such as $\theta \leftarrow \theta - \eta\,\partial L/\partial\theta$ to reduce the loss. Here we only compute and inspect the gradients.